## Dataerai notebook and neural-network provenance

Authenticate once with `dataerai auth login --device --client-id dataerai-mobile --server https://beta.dataerai.com`. The next cell starts `%dataerai --trace` and the M3 artifact publisher. Together they preserve every execution; upload the source notebook; reuse raw datasets; publish figures, movies, changed HDF5/CSV data, PyTorch checkpoints, loss histories, and manifests; and link those products to the notebook run and source data. Output folders are mirrored as nested Dataerai collections. Set `DATAERAI_DESTINATION_COLLECTION_PATH` before launching Jupyter to override the default destination.


In [ ]:
import os as _dataerai_os
import subprocess as _dataerai_subprocess
import sys as _dataerai_sys

_dataerai_subprocess.run(
    [
        _dataerai_sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--pre",
        "dataerai-cli-beta==0.1.54",
        "dataerai-sdk[notebook,nn-pytorch]==0.2.0b52",
    ],
    check=True,
)

try:
    from m3_learning.artifacts import start_dataerai_artifact_publishing
except ImportError:
    _dataerai_subprocess.run(
        [
            _dataerai_sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "--no-deps",
            "git+https://github.com/m3-learning/"
            "m3_learning.git@codex/dataerai-notebook-training-provenance"
            "#subdirectory=m3_learning",
        ],
        check=True,
    )
    from m3_learning.artifacts import start_dataerai_artifact_publishing

DATAERAI_NOTEBOOK = 'fast_force_mapping.ipynb'
DATAERAI_NOTEBOOK_TITLE = 'fast force mapping'
DATAERAI_DESTINATION_COLLECTION_PATH = _dataerai_os.environ.get(
    "DATAERAI_DESTINATION_COLLECTION_PATH",
    'M3 Learning / Notebook Provenance / Testing',
)

DATAERAI_PROVENANCE_ROOT_PATH = (
    f"{DATAERAI_DESTINATION_COLLECTION_PATH} / fast_force_mapping"
)
DATAERAI_EXECUTION_COLLECTION_PATH = (
    f"{DATAERAI_PROVENANCE_ROOT_PATH} / Executions"
)

%load_ext dataerai.magics
%dataerai --request-timeout 120 --trace --notebook "$DATAERAI_NOTEBOOK" --title "$DATAERAI_NOTEBOOK_TITLE" "$DATAERAI_EXECUTION_COLLECTION_PATH"

_dataerai_os.environ["DATAERAI_NOTEBOOK_TRACE_RUN_ID"] = dataerai_session.trace_run_id
_dataerai_os.environ["DATAERAI_NOTEBOOK_COLLECTION_PATH"] = dataerai_session.collection_path
dataerai_artifacts = start_dataerai_artifact_publishing(
    dataerai_session,
    DATAERAI_NOTEBOOK,
    shell=get_ipython(),
    provenance_root_path=DATAERAI_PROVENANCE_ROOT_PATH,
)


In [1]:
import sys
sys.path.append('../../')
sys.path.append('C:\codes\m3_learning\m3_learning\src')

In [3]:
from m3_learning.nn.Fitter1D.Fitter1D import Multiscale1DFitter, Model
import torch


In [15]:
from m3_learning.be.loop_fitter import loop_fitting_function_torch


model_ = Multiscale1DFitter(loop_fitting_function_torch, # function 
                            x_data, # x data
                            1, # input channels
                            9, # output parameters
                            dataset.SHO_scaler, 
                            postprocessor)

In [4]:
def hertz_model(V, y, device = "cuda"):
    V = torch.tensor(V, dtype=torch.float64, device=device)

    try:
        y = torch.from_numpy(y).to(device)
        if len(y.shape) == 1:
            y = torch.unsqueeze(y, 0)
    except:
        pass
    
    E_star, R = y[:, 0], y[:, 1]
    d = V.type(torch.float64)

    F_hertz = (4 / 3) * E_star * torch.sqrt(R) * torch.pow(d, 1.5)
    return F_hertz

In [ ]:
np.

## Finish the Dataerai provenance record

This cell first uploads captured figures, movies, changed HDF5/CSV files, and model artifacts. It then publishes this run's distinct notebook execution log and its `records_telemetry` relationships.


In [ ]:
dataerai_artifact_result = dataerai_artifacts.finish()
%dataerai --finish
